# Agents

## Load the Environment

Load the environment:

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Defining the Agent

Import the necessary classes:

In [2]:
from pydantic import BaseModel, Field

from limin import ModelConfiguration, Tool, Agent

In order to define an agent we need to define three things:

- the model configuration for the agent
- the initial system prompt for the agent
- the tools the agent can use

Defining the model configuration and system prompt is easy:

In [3]:
model_configuration = ModelConfiguration(model="gpt-4o", temperature=1.0)
system_prompt = "You are a helpful assistant"

The tools are a bit trickier. For every tool we need to define a unique tool name, a tool description, a pydantic Model that describes the parameters the tool can take and an execution function that will be called when the agent decides to execute the tool.

We will define 2 tools: a simple calculator tool which can add, substract, multiply and divide two numbers and a simple weather tool which will return a mock temperature.

We begin by defining the calculator tool:

In [4]:
class GetCalculatorParameters(BaseModel):
    operation: str = Field(
        description="Operation to perform - with +, -, *, / (e.g. 2*2)"
    )
    operator1: int = Field(description="First number to perform the operation on")
    operator2: int = Field(description="Second number to perform the operation on")


def get_calculator_exec(operation: str, operator1: int, operator2: int) -> str:
    if operation == "+":
        return str(operator1 + operator2)
    elif operation == "-":
        return str(operator1 - operator2)
    elif operation == "*":
        return str(operator1 * operator2)
    elif operation == "/":
        if operator2 == 0:
            return "Error: Division by zero"
        return str(operator1 / operator2)
    else:
        return f"Error: Unknown operation '{operation}'"


get_calculator_tool = Tool(
    name="get_calculator",
    description="Get the result of the operation on the list of numbers",
    parameters=GetCalculatorParameters,
    exec_fn=get_calculator_exec,
)

Note that the descriptions and the execution function must match. For example, the `GetCalculatorParameters` specifies that the operation must be either `+`, `-`, `*` or `/` and not `add`, `substract`, `multiply` or `divide`.

We define the mock weather tool in a similar fashion:

In [5]:
class GetWeatherParameters(BaseModel):
    location: str = Field(description="City and country e.g. Munich, Germany")


def get_weather_exec(location: str) -> str:
    # Mock weather function - in reality this would call a weather API
    return "22°C"


get_weather_tool = Tool(
    name="get_weather",
    description="Get current temperature for provided location in celsius.",
    parameters=GetWeatherParameters,
    exec_fn=get_weather_exec,
)

Now we can speficy the agent:

In [6]:
agent = Agent(
    system_prompt="You are a helpful assistant.",
    tools=[get_weather_tool, get_calculator_tool],
    model_configuration=model_configuration,
)

## Using the Agent

To use the agent we can call the `agent.process` method. Lets start with a simple request:

In [7]:
messages = await agent.process("How are you today")

The agent will return the newly created messages and automatically append them to the conversation. In the simplest case, the agent will append the user message (your request) and the assistant message that responds to the user message:

In [8]:
print(messages)

[UserMessage(role='user', content='How are you today'), AssistantMessage(role='assistant', content="Thank you for asking! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?", tool_calls=None)]


In [9]:
print(messages[-1])

role='assistant' content="Thank you for asking! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?" tool_calls=None


You can get the conversation so far by accessing the `agent.conversation` field which is just a regular `Conversation` object:

In [10]:
print(agent.conversation.to_markdown())

## System 
You are a helpful assistant.

## User 
How are you today

## Assistant 
Thank you for asking! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?


Now let's make the request a bit more complex and ask about the weather in Paris. This will trigger a tool call:

In [11]:
weather_messages = await agent.process("What's the weather like in Paris today?")

Lets see how the messages look like:

In [12]:
print(weather_messages)

[UserMessage(role='user', content="What's the weather like in Paris today?"), AssistantMessage(role='assistant', content=None, tool_calls=[ToolCall(id='call_TulPrFJorXebmUw3PsqEDb8I', name='get_weather', arguments={'location': 'Paris, France'})]), ToolMessage(role='tool', tool_call_id='call_TulPrFJorXebmUw3PsqEDb8I', content='22°C'), AssistantMessage(role='assistant', content='The weather in Paris today is 22°C. If you need more details or updates, feel free to ask!', tool_calls=None)]


This is a bit more complicated than before. First, we again have your request given as a user message. However, the assistant message does not have any content - instead it contains a list of tool calls together with their arguments:

In [13]:
print(weather_messages[1].tool_calls)

[ToolCall(id='call_TulPrFJorXebmUw3PsqEDb8I', name='get_weather', arguments={'location': 'Paris, France'})]


It is important to understand that no tools have been called yet. Instead, the assistant message contains all the tools that should be called by the agent. The agent then calls the tools and generates a `ToolMessage` containing the result of the tool call (specifically the result returned by the `exec_fn` function of the tool):

In [14]:
print(weather_messages[2])

role='tool' tool_call_id='call_TulPrFJorXebmUw3PsqEDb8I' content='22°C'


The final message is the final response of the assistant based on the user request and the tool call result contained in the `ToolMessage`:

In [15]:
print(weather_messages[3])

role='assistant' content='The weather in Paris today is 22°C. If you need more details or updates, feel free to ask!' tool_calls=None


We can once again print the entire conversation so far:

In [16]:
print(agent.conversation.to_markdown())

## System 
You are a helpful assistant.

## User 
How are you today

## Assistant 
Thank you for asking! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?

## User 
What's the weather like in Paris today?

## Assistant 
None

## Tool 
22°C

## Assistant 
The weather in Paris today is 22°C. If you need more details or updates, feel free to ask!


Let's do this again. This time we will provide a request that will force the agent to call the calculator tool:

In [17]:
math_messages = await agent.process("What's the result of 2+2?")

In [18]:
print(math_messages)

[UserMessage(role='user', content="What's the result of 2+2?"), AssistantMessage(role='assistant', content=None, tool_calls=[ToolCall(id='call_xMDe98IjbQDFumVpdof8rDMD', name='get_calculator', arguments={'operation': '+', 'operator1': 2, 'operator2': 2})]), ToolMessage(role='tool', tool_call_id='call_xMDe98IjbQDFumVpdof8rDMD', content='4'), AssistantMessage(role='assistant', content='The result of \\(2 + 2\\) is 4.', tool_calls=None)]


This is very similar to `weather_messages`. We have a user message, followed by the assistant message containing the tool calls, followed by the tool call result, followed by the final assistant message.

Let's print the entire conversation so far:

In [19]:
print(agent.conversation.to_markdown())

## System 
You are a helpful assistant.

## User 
How are you today

## Assistant 
Thank you for asking! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with whatever you need. How can I assist you today?

## User 
What's the weather like in Paris today?

## Assistant 
None

## Tool 
22°C

## Assistant 
The weather in Paris today is 22°C. If you need more details or updates, feel free to ask!

## User 
What's the result of 2+2?

## Assistant 
None

## Tool 
4

## Assistant 
The result of \(2 + 2\) is 4.
